# VinBigData Chest X-ray Abnormalities Detection — Exploratory Data Analysis

This notebook explores the competition data with one goal: **every analysis should feed a concrete pipeline decision**.
Each section opens with the key insight (based on a full run over all 15,000 training images), followed by the code that reproduces it.

**Run order:** the setup cells (Section 0) must run first. Reading the DICOM headers of all 18,000 files takes a few minutes.

### Contents
0. Setup — imports, data loading, DICOM metadata, box geometry
1. Dataset overview — files, annotation structure, class distribution
2. Image-level labels are perfectly clean
3. Three radiologists labeled 94% of the abnormal images
4. Long tail and per-class agreement
5. Two "schools" of box drawing
6. Box-level agreement is bimodal
7. Box size: rare ≠ small
8. Resolution matters for only 3 classes — but a lot
9. Strong anatomical location per class
10. Class co-occurrence — clinical links and semantic hierarchy
11. Metadata: no train/test shift, no usable patient information
12. Visual examples — one pathology, three "languages"
13. Summary — decisions and open questions

## 0. Setup

In [ ]:
import os, glob, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pydicom
from joblib import Parallel, delayed
from tqdm.auto import tqdm

try:  # pydicom >= 3
    from pydicom.pixels import apply_voi_lut
except ImportError:  # pydicom 2.x
    from pydicom.pixel_data_handlers.util import apply_voi_lut

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

DATA_DIR = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection'
OUT_DIR = '/kaggle/working'
NO_FINDING_ID = 14

In [ ]:
# Load the annotations: one row = one box (or one "No finding" vote) by one radiologist
train = pd.read_csv(f'{DATA_DIR}/train.csv')

abn = train[train.class_id != NO_FINDING_ID].copy()   # abnormality boxes only
img_has_abnormal = train.groupby('image_id').class_id.apply(lambda s: (s != NO_FINDING_ID).any())
normal_ids = img_has_abnormal[~img_has_abnormal].index

print('annotations:', len(train), '| images:', train.image_id.nunique(), '| boxes:', len(abn))

In [ ]:
# DICOM headers only (no pixels) -> image size, photometric interpretation, patient fields
def read_meta(path, split):
    d = pydicom.dcmread(path, stop_before_pixels=True)
    return dict(image_id=os.path.basename(path)[:-6], split=split,
                rows=int(d.Rows), cols=int(d.Columns),
                photometric=d.PhotometricInterpretation,
                sex=getattr(d, 'PatientSex', None), age=getattr(d, 'PatientAge', None))

META_CACHE = f'{OUT_DIR}/meta.csv'
if os.path.exists(META_CACHE):
    meta = pd.read_csv(META_CACHE)
else:
    paths = ([(p, 'train') for p in glob.glob(f'{DATA_DIR}/train/*.dicom')] +
             [(p, 'test') for p in glob.glob(f'{DATA_DIR}/test/*.dicom')])
    meta = pd.DataFrame(Parallel(n_jobs=-1)(delayed(read_meta)(p, s) for p, s in tqdm(paths)))
    meta.to_csv(META_CACHE, index=False)
print(meta.split.value_counts().to_dict())

In [ ]:
# Box geometry, both in pixels and normalized by the image size
abn = abn.merge(meta[['image_id', 'rows', 'cols']], on='image_id', how='left')
abn['w'] = abn.x_max - abn.x_min
abn['h'] = abn.y_max - abn.y_min
abn['area'] = abn.w * abn.h
abn['area_frac'] = abn.area / (abn.rows * abn.cols)                       # share of image area
abn['side_frac'] = np.sqrt(abn.area_frac)                                 # "typical side" relative to image
abn['min_side_frac'] = np.minimum(abn.w / abn.cols, abn.h / abn.rows)     # the thin dimension
abn['xc'] = (abn.x_min + abn.x_max) / 2 / abn.cols                        # normalized box center
abn['yc'] = (abn.y_min + abn.y_max) / 2 / abn.rows

## 1. Dataset overview
- **15,000 train** and **3,000 test** DICOM files, plus `train.csv` and `sample_submission.csv`.
- `train.csv` has **67,914 rows** with 8 columns: `image_id, class_name, class_id, rad_id, x_min, y_min, x_max, y_max`.
- Each row is **one radiologist's vote**: either a box for one of 14 abnormality classes, or a "No finding" row (`class_id = 14`) with empty coordinates.
  That is why only 36,096 rows have coordinates.
- Raw counts are dominated by "No finding" (31,818 rows). Among abnormalities, Aortic enlargement (7,162) and Cardiomegaly (5,427) lead, while Atelectasis (279) and Pneumothorax (226) are at the bottom.

**Note:** row counts mix "images" and "boxes" — a class with many boxes may still appear in few images. Section 4 separates the two.

In [ ]:
# Competition files
for name in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, name)
    print(f'  {name}/  ({len(os.listdir(p))} files)' if os.path.isdir(p) else f'  {name}')

In [ ]:
# Annotation table structure: coordinates are NaN for "No finding" rows
train.info()
train.head(10)

In [ ]:
# Distribution of the 14 abnormality classes + "No finding" (row level)
class_counts = train.class_name.value_counts()

plt.figure(figsize=(11, 6))
sns.barplot(x=class_counts.values, y=class_counts.index, hue=class_counts.index, palette='viridis', legend=False)
plt.title('Number of annotations per class')
plt.xlabel('Count'); plt.ylabel('')
plt.tight_layout(); plt.show()
class_counts

## 2. Image-level labels are perfectly clean
- **70.7%** of the images are normal (10,606 of 15,000); 4,394 are abnormal.
- **No image is disputed**: whenever an image is abnormal, all 3 radiologists drew at least one box on it. No image mixes "No finding" votes with boxes.
- A radiologist draws on average **2.7 boxes per abnormal image** (median 2, max 28). Summed over 3 radiologists: median 7, max 57 boxes per image.

**Implication:** the disagreement is about *what and where*, never about *whether*. A binary normal/abnormal classifier would be trained on reliable labels.

In [ ]:
n_total = len(img_has_abnormal)
print(f'normal images:   {len(normal_ids)} ({len(normal_ids) / n_total:.1%})')
print(f'abnormal images: {n_total - len(normal_ids)} ({1 - len(normal_ids) / n_total:.1%})')

# Is any image both "No finding" (for some radiologist) and boxed (by another)?
nf_ids = set(train.loc[train.class_id == NO_FINDING_ID, 'image_id'])
print('images with both "No finding" and a box:', len(nf_ids & set(abn.image_id)))

# How many radiologists drew boxes on each abnormal image, and how many boxes each
per_rad_img = abn.groupby(['image_id', 'rad_id']).size()
print('# rads that drew boxes, per abnormal image:', per_rad_img.groupby('image_id').size().value_counts().to_dict())
print('boxes per rad per image: mean %.2f | median %d | max %d'
      % (per_rad_img.mean(), per_rad_img.median(), per_rad_img.max()))

In [ ]:
# Total boxes per abnormal image (all radiologists together)
boxes_per_image = abn.groupby('image_id').size()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].pie([n_total - len(normal_ids), len(normal_ids)], labels=['Abnormal', 'No finding'],
          autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'], startangle=90)
ax[0].set_title('Image level: abnormal vs. no finding')
boxes_per_image.hist(bins=50, color='#9b59b6', ax=ax[1])
ax[1].set_title('Abnormality boxes per abnormal image (3 radiologists combined)')
ax[1].set_xlabel('# boxes'); ax[1].set_ylabel('# images')
plt.tight_layout(); plt.show()
boxes_per_image.describe().round(2)

## 3. Three radiologists labeled 94% of the abnormal images
- Every image was read by exactly **3 radiologists**, drawn from a pool of 17 (R1–R17).
- **R8, R9, R10** together labeled 4,146 of the 4,394 abnormal images — the same trio on each of them.
- The remaining 248 abnormal images were labeled by triplets drawn from **R11–R17**.
- **R1–R7** labeled almost only normal images (R2 has a single abnormal image).

**Implication:** in practice the detector learns the style of the R8–R10 trio. We do not know which 5 radiologists produced the test-set consensus.

In [ ]:
# Workload per radiologist and number of radiologists per image
rads_per_image = train.groupby('image_id').rad_id.nunique()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
train.rad_id.value_counts().plot.bar(ax=ax[0], color='#34495e')
ax[0].set_title('Annotation rows per radiologist'); ax[0].set_ylabel('# rows')
rads_per_image.value_counts().sort_index().plot.bar(ax=ax[1], color='#3498db')
ax[1].set_title('Radiologists per image'); ax[1].set_xlabel('# radiologists'); ax[1].set_ylabel('# images')
plt.tight_layout(); plt.show()

In [ ]:
# Images read by each radiologist, split by image status
ra = train.drop_duplicates(['image_id', 'rad_id'])
ra = ra.assign(kind=np.where(ra.image_id.isin(normal_ids), 'normal', 'abnormal'))
pd.crosstab(ra.rad_id, ra.kind).sort_values('abnormal', ascending=False)

## 4. Long tail and per-class agreement
- At image level: Aortic enlargement appears in 3,067 images vs. Pneumothorax in 96 — a **32× gap**.
- Agreement (≥ 2 radiologists marked the class in the image) ranges from **79%** (Cardiomegaly) down to **32%** (Other lesion).
- **Pneumothorax is rare but reliable**: 45 of its 96 images were marked by all 3 radiologists.
- **Atelectasis and Consolidation**: in most of their images only one radiologist marked them — keeping only agreed findings would shrink them roughly 3×.

**Implication:** mAP weights every class equally, so the rare classes drive the score. We need Multilabel Stratified K-Fold and class-aware sampling.

In [ ]:
img_cls = abn.groupby('class_name').agg(boxes=('image_id', 'size'), images=('image_id', 'nunique'))
pair = abn.groupby(['class_name', 'image_id']).rad_id.nunique()      # radiologists per (class, image)
img_cls['images_2plus_rads'] = pair[pair >= 2].groupby('class_name').size()
img_cls['images_3_rads'] = pair[pair == 3].groupby('class_name').size()
img_cls = img_cls.fillna(0).astype(int).sort_values('images', ascending=False)
img_cls['pct_agreed'] = (100 * img_cls.images_2plus_rads / img_cls.images).round(1)
display(img_cls)

img_cls[['images', 'images_2plus_rads', 'images_3_rads']].plot.barh(figsize=(10, 6), logx=True)
plt.title('Images per class (log scale)'); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 5. Two "schools" of box drawing
- The R8–R10 trio is **highly consistent**: median side ratio (box side / image side) of 0.11–0.12 for Aortic enlargement, 0.24–0.25 for Cardiomegaly, 0.04 for Nodule/Mass.
- R11–R17 draw **much larger** boxes: Aortic enlargement for R12 = 0.33 (3× the side, 9× the area); Cardiomegaly 0.32–0.42.
- R8 marks every nodule separately: 2.6 Nodule/Mass boxes per image vs. 1.6–1.7 for R9/R10.

**Implication:** the 248 images labeled by R11–R17 are written in a different "language" → conflicting targets. We need to decide how to handle them.
(Note: some of R11–R17 have few images, so their medians are noisy — but the direction is consistent.)

In [ ]:
# Per radiologist x class: how many boxes per image, and how large
bpc = abn.groupby(['rad_id', 'image_id', 'class_name']).size().rename('n').reset_index()
n_piv = bpc.pivot_table(index='rad_id', columns='class_name', values='n', aggfunc='mean')
s_piv = abn.pivot_table(index='rad_id', columns='class_name', values='side_frac', aggfunc='median')

fig, ax = plt.subplots(1, 2, figsize=(26, 7))
sns.heatmap(n_piv, annot=True, fmt='.1f', cmap='mako_r', cbar=False, ax=ax[0])
ax[0].set_title('Mean boxes per image (per radiologist & class)')
sns.heatmap(s_piv, annot=True, fmt='.2f', cmap='rocket_r', cbar=False, ax=ax[1])
ax[1].set_title('Median box side / image side')
plt.tight_layout(); plt.show()

## 6. Box-level agreement is bimodal
For every box: the best IoU against a box from a **different** radiologist, of the same class, in the same image.
- A sharp peak at **IoU = 0** (~2,450 boxes) and a "hill" between **0.6–0.9** (peak ~0.78), with almost nothing in between.
- In other words: when radiologists agree, they agree well; when they don't, they marked a different finding or a different place — not a small shift.
- The "human ceiling" at the metric threshold (IoU ≥ 0.4): ~99% for Cardiomegaly, but only **57%** for Pleural thickening and 59% for Other lesion.

**Implication:** the IoU threshold for WBF (0.3–0.5) barely matters. The real decision is what to do with boxes marked by a single radiologist.

In [ ]:
def iou_matrix(a, b):
    # Pairwise IoU between two arrays of [x_min, y_min, x_max, y_max] boxes
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

rec = []
for (iid, cname), g in tqdm(abn.groupby(['image_id', 'class_name'])):
    rads = g.rad_id.values
    n_rads = len(set(rads))
    if n_rads < 2:
        best = np.full(len(g), np.nan)
    else:
        boxes = g[['x_min', 'y_min', 'x_max', 'y_max']].values
        ious = iou_matrix(boxes, boxes)
        ious[rads[:, None] == rads[None, :]] = -1          # ignore same-radiologist pairs
        best = ious.max(1)
    rec += [dict(class_name=cname, n_rads=n_rads, best_iou=v) for v in best]
iou_df = pd.DataFrame(rec)

print('% of boxes by # radiologists who marked that class in the image:')
display((iou_df.groupby('class_name').n_rads.value_counts(normalize=True).unstack().fillna(0) * 100).round(1))

m = iou_df.dropna()
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
m.best_iou.hist(bins=50, ax=ax[0]); ax[0].axvline(0.4, color='r', ls='--', label='metric IoU=0.4'); ax[0].legend()
ax[0].set_title('Best IoU with another radiologist')
m.assign(ok=m.best_iou >= 0.4).groupby('class_name').ok.mean().sort_values().plot.barh(ax=ax[1])
ax[1].set_title('Share of boxes matched by another rad at IoU >= 0.4')
plt.tight_layout(); plt.show()

## 7. Box size: rare ≠ small
- In raw pixels, boxes are large overall (median ~323×320 px on ~2,800×2,400 images), and the log-area histogram is **bimodal**.
- The second (large) mode is mostly **Cardiomegaly**: 96% of its boxes fall there, and it makes up 44% of that mode.
- Normalized by image area — **smallest**: Nodule/Mass (median ~0.14% of the image), Calcification, Pleural thickening. **Largest**: ILD, Pneumothorax, Cardiomegaly (~6–7%).
- **Calcification** is the outlier: both relatively rare (452 images) and small ⇒ probably the hardest class.

**Implication:** two independent axes — size ⇒ resolution; rarity ⇒ sampling. Both can be handled within a single model.

In [ ]:
# Raw box dimensions in pixels
display(abn[['w', 'h', 'area']].describe().round(0))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(abn.w, bins=60, color='#1abc9c'); axes[0].set_title('Box width (px)')
axes[1].hist(abn.h, bins=60, color='#e67e22'); axes[1].set_title('Box height (px)')
axes[2].hist(np.log1p(abn.area), bins=60, color='#c0392b'); axes[2].set_title('log(1 + area)')
axes[2].axvline(12.3, color='k', ls='--', label='mode split'); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Which classes populate the large-area mode of the histogram?
abn['big_mode'] = np.log1p(abn.area) > 12.3
share = abn.groupby('class_name').big_mode.agg(n_big='sum', pct_big='mean')
share['pct_of_big_mode'] = share.n_big / share.n_big.sum()
display(share.sort_values('n_big', ascending=False).round(2))

# Box area relative to image area, per class
order = abn.groupby('class_name').area_frac.median().sort_values().index
plt.figure(figsize=(12, 7))
sns.boxplot(data=abn, y='class_name', x='area_frac', order=order, fliersize=1)
plt.xscale('log'); plt.title('Box area / image area (log)'); plt.tight_layout(); plt.show()

## 8. Resolution matters for only 3 classes — but a lot
Share of boxes whose typical side is < 16 px after resizing the image:

| Class | 512 | 768 | 1024 |
|---|---|---|---|
| Nodule/Mass | 42.2% | 21.5% | 10.1% |
| Calcification | 19.0% | 7.2% | 1.6% |
| Other lesion | 10.8% | 4.9% | 1.8% |

All other classes are ≈ 0 already at 768. Pleural thickening is **not** too thin (0.2% below 8 px at 768).

**Implication:** 512 is ruled out. 768 is the minimum; 1024 is preferred.

In [ ]:
res_list = [512, 768, 1024]
# "Tiny": typical side (sqrt of area) below 16 px after resizing
tiny = pd.DataFrame({r: abn.assign(s=abn.side_frac * r).groupby('class_name').s.apply(lambda s: (s < 16).mean() * 100)
                     for r in res_list}).round(1)
# "Thin": the shorter side below 8 px after resizing
thin = pd.DataFrame({r: abn.assign(s=abn.min_side_frac * r).groupby('class_name').s.apply(lambda s: (s < 8).mean() * 100)
                     for r in res_list}).round(1)
display(tiny.sort_values(512, ascending=False).add_suffix('px: %side<16'))
display(thin.sort_values(512, ascending=False).add_suffix('px: %thin<8'))

## 9. Strong anatomical location per class
(Radiological convention: the right side of the image is the patient's left side.)
- **Aortic enlargement** (aortic arch) and **Cardiomegaly**: 96% on the right side of the image.
- **Pleural thickening**: lung apices. **Pleural effusion**: bottom (gravity). **Pulmonary fibrosis**: upper lobes.
- **Calcification**: a hot spot near the aorta (aortic-arch calcification). **Other lesion**: spread across the whole image (a "catch-all" class).

**Implication:** no horizontal flip in the baseline — it creates impossible anatomy and weakens the strongest prior we have.

In [ ]:
# Normalized coverage map: accumulate every box on a G x G grid, per class
G = 64
heat = {}
for cname, g in abn.groupby('class_name'):
    h = np.zeros((G, G))
    for x0, y0, x1, y1, W, H in g[['x_min', 'y_min', 'x_max', 'y_max', 'cols', 'rows']].values:
        h[int(y0 / H * G):int(np.ceil(y1 / H * G)), int(x0 / W * G):int(np.ceil(x1 / W * G))] += 1
    heat[cname] = h

fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for ax, (cname, h) in zip(axes.flat, heat.items()):
    ax.imshow(h / h.max(), cmap='hot'); ax.set_title(cname, fontsize=10)
for ax in axes.flat: ax.axis('off')
plt.suptitle('Normalized box coverage per class'); plt.tight_layout(); plt.show()

# Horizontal position of box centers
abn.groupby('class_name').xc.agg(mean_xc='mean', frac_image_left=lambda s: (s < 0.5).mean()).round(2)

## 10. Class co-occurrence — clinical links and semantic hierarchy
- **Consolidation → Lung Opacity: 0.84** — Consolidation is a type of opacity; the same region gets both labels.
- **Cardiomegaly → Aortic enlargement: 0.86**, **Pleural effusion → Pleural thickening: 0.81** — clinical association.
- **Trap:** a high P(col | row) does not imply a real association if the column class is simply common (Aortic enlargement ≈ 70% of abnormal images).
  So we also show **lift** = P(col | row) / P(col): above 1 = real association, below 1 = none.

**Implication:** use **per-class** NMS, not class-agnostic — otherwise Lung Opacity would suppress an almost identical Consolidation box.

In [ ]:
onehot = pd.crosstab(abn.image_id, abn.class_name).clip(upper=1)   # image x class presence
co = onehot.T @ onehot
cond = co.div(pd.Series(np.diag(co), index=co.index), axis=0)       # P(col | row)
lift = cond.div(onehot.mean(), axis=1)                              # / P(col)
lift = lift.mask(np.eye(len(lift), dtype=bool))                     # hide diagonal

fig, ax = plt.subplots(1, 2, figsize=(28, 10))
sns.heatmap(cond, annot=True, fmt='.2f', cmap='Blues', cbar=False, ax=ax[0])
ax[0].set_title('P(column present | row present)')
sns.heatmap(lift, annot=True, fmt='.1f', cmap='RdBu_r', center=1, cbar=False, ax=ax[1])
ax[1].set_title('Lift = P(col | row) / P(col)   (>1 = real association)')
plt.tight_layout(); plt.show()

## 11. Metadata: no technical train/test shift, no usable patient information
- **MONOCHROME1:** 17.6% in train, 18% in test ⇒ inversion is mandatory, and the ratio is identical.
- **Image size:** nearly identical distributions (median ~2,840×2,370). Some small images exist (~900 px).
- **No patient / study / device identifiers** (only 23 technical fields survived de-identification) ⇒ duplicates can only be detected by hashing the images.
- **Age:** usable in only ~23% of images (25% in train, 16% in test), and includes children (ages 2–4).
  **Sex:** empty or "O" in about half. ⇒ neither goes into the model.
- The missing rate of age/sex **differs between train and test** — a hint of a slightly different source mix. Watch for a CV↔LB gap.
- `WindowCenter/Width` are missing in 3.6% of files ⇒ check their contrast before the PNG conversion.

In [ ]:
# Photometric interpretation and sex, normalized per split
for c in ['photometric', 'sex']:
    display(pd.crosstab(meta[c].fillna('NA'), meta.split, normalize='columns').round(3))

# Image size per split
display(meta.groupby('split')[['rows', 'cols']].describe().T.round(0))
sns.scatterplot(data=meta.sample(4000, random_state=0), x='cols', y='rows', hue='split', s=10, alpha=.4)
plt.title('Image size: train vs test'); plt.show()

In [ ]:
# Age is stored as a string such as '054Y'; extract the number and keep plausible values only
meta['age_years'] = pd.to_numeric(meta.age.astype(str).str.extract(r'(\d+)')[0], errors='coerce')
valid_age = meta.age_years.between(1, 100)
print('missing:', meta.age_years.isna().mean().round(3), '| zero:', (meta.age_years == 0).mean().round(3))
print('valid age share by split:', valid_age.groupby(meta.split).mean().round(3).to_dict())
display(meta[valid_age].groupby('split').age_years.describe().round(1))

In [ ]:
# Which DICOM header fields survived de-identification (sample of 500 train files)
sample = glob.glob(f'{DATA_DIR}/train/*.dicom')[:500]
cnt = Counter(el.keyword for p in sample for el in pydicom.dcmread(p, stop_before_pixels=True))
(pd.Series(cnt) / len(sample)).sort_values(ascending=False).round(3)

## 12. Visual examples — one pathology in three "languages", two box styles
- **Left:** the image with the most boxes — one radiologist draws a huge box per lung, another marks dozens of individual nodules, and there is also ILD.
  WBF will not merge a huge box with a small nodule (IoU ≈ 0) ⇒ each "language" stays a single-radiologist box.
- **Right:** Aortic enlargement on an image labeled by R12 — a much wider box than the R8–R10 style.

Colors represent **radiologists**, not boxes.

In [ ]:
def read_dicom(path):
    # Apply the VOI LUT, invert MONOCHROME1 images, and scale to uint8
    d = pydicom.dcmread(path)
    data = apply_voi_lut(d.pixel_array, d).astype(np.float32)
    if d.PhotometricInterpretation == 'MONOCHROME1':
        data = data.max() - data
    data = (data - data.min()) / (data.max() - data.min() + 1e-6)
    return (data * 255).astype(np.uint8)

def show_by_rad(image_id, ax, classes=None, step=3):
    # Draw the boxes of one image, colored by radiologist; `step` downsamples for speed
    img = read_dicom(f'{DATA_DIR}/train/{image_id}.dicom')[::step, ::step]
    ax.imshow(img, cmap='gray'); ax.axis('off')
    g = abn[abn.image_id == image_id]
    if classes is not None:
        g = g[g.class_name.isin(classes)]
    palette = dict(zip(sorted(g.rad_id.unique()), ['red', 'lime', 'cyan', 'yellow', 'magenta']))
    for _, r in g.iterrows():
        ax.add_patch(patches.Rectangle((r.x_min / step, r.y_min / step), r.w / step, r.h / step,
                                       fill=False, ec=palette[r.rad_id], lw=1.5))
    handles = [patches.Patch(color=c, label=rad) for rad, c in palette.items()]
    ax.legend(handles=handles, loc='lower right', fontsize=9)

busiest = abn.image_id.value_counts().index[0]
r12_aortic = abn[(abn.rad_id == 'R12') & (abn.class_name == 'Aortic enlargement')].image_id.iloc[0]

fig, ax = plt.subplots(1, 2, figsize=(20, 10))
show_by_rad(busiest, ax[0]); ax[0].set_title(f'Busiest image ({busiest[:10]}…) — all classes')
show_by_rad(r12_aortic, ax[1], classes=['Aortic enlargement', 'Cardiomegaly'])
ax[1].set_title(f'R11–R17 style ({r12_aortic[:10]}…) — Aortic enlargement & Cardiomegaly')
plt.tight_layout(); plt.show()

## 13. Summary

### Decisions already justified by the EDA
| Decision | Based on |
|---|---|
| Input resolution ≥ 768, preferably 1024 | Section 8 |
| No horizontal flip in the baseline | Section 9 |
| Per-class NMS | Section 10 |
| Invert MONOCHROME1 images | Section 11 |
| Binary normal/abnormal classifier trained on clean labels | Section 2 |
| Multilabel Stratified K-Fold + sampling for rare classes | Section 4 |
| Do not use age/sex | Section 11 |

### Open questions (for the pipeline stage)
1. **How to build the validation ground truth** so that it mimics a 5-radiologist consensus? (Sections 3, 6)
2. **What to do with single-radiologist boxes** — keep, down-weight, or drop? (Sections 4, 6) → experiments A vs. B.
3. **What to do with the 248 images labeled by R11–R17**, which follow a different box style? (Sections 3, 5)
4. **Duplicates / same patient** — to be detected with a perceptual hash during the PNG conversion. (Section 11)
5. **Images without a VOI LUT (3.6%)** — how to normalize them during conversion. (Section 11)